# Session 3 — RAG in Production

## From toy pipeline to production system

The [RAG Fundamentals](../rag-fundamentals/rag-fundamentals.html) notebook built a working pipeline: chunk -> embed ->
cosine search -> stuff context -> generate. That pipeline works on 4 documents. It breaks down once
the corpus is real:

- **Pure vector search misses exact matches.** Embeddings capture meaning, not keywords — a query for an
  error code or product SKU can score *worse* than a semantically related but wrong chunk.
- **Nobody measures whether retrieval is actually good**, so quality regressions ship silently.
- **`O(n)` cosine scan over every vector doesn't scale** past a few thousand chunks.
- **The index goes stale** the moment a source document changes, and nothing tells you.

This notebook covers the four levers used to fix each of these: hybrid search, evaluation, ANN
indexing, and freshness.

In [1]:
import os
import re
import math
from collections import Counter

import numpy as np
from google import genai
from google.genai import types

from dotenv import load_dotenv
load_dotenv()

api_key = os.environ.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

EMBED_MODEL = "gemini-embedding-001"
CHAT_MODEL = "gemini-2.5-flash-lite"

def embed(texts):
    resp = client.models.embed_content(model=EMBED_MODEL, contents=texts)
    return np.array([e.values for e in resp.embeddings])

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# small demo corpus: mix of exact-keyword-sensitive and semantic content
corpus = [
    "Error code E-4521 means the payment gateway timed out; retry the charge after 30 seconds.",
    "The checkout service occasionally fails to reach the payment provider under high load.",
    "Our return policy allows refunds within 30 days of purchase with a valid receipt.",
    "Photosynthesis converts light energy into chemical energy stored in glucose.",
    "Restarting the payment-gateway pod usually clears transient E-4521 errors.",
    "The French Revolution began in 1789 and led to the end of the monarchy.",
]

## 1. Retrieval quality: hybrid search + reranking

**Why pure vector search fails here:** a query for `"E-4521"` should retrieve the two chunks that
literally mention that error code. But an embedding model represents *meaning*, and "error code" +
some digits doesn't have strong semantic neighbors — a chunk about "checkout service failing under
load" can out-score it on cosine similarity even though it never mentions the code.

**Fix: hybrid search.** Run two retrievers in parallel — a lexical one (BM25, keyword-based) and the
vector one from before — then merge their rankings with **Reciprocal Rank Fusion (RRF)**: each chunk
gets `1 / (k + rank)` from each retriever, summed. A chunk that ranks well in *either* list surfaces;
one that ranks well in *both* wins outright. No score normalization needed, which is why RRF is the
default fusion method in most production hybrid-search stacks.

BM25 from scratch below — same "no black box" treatment as the vector store.

In [2]:
class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.docs = [self._tokenize(d) for d in docs]
        self.doc_lens = [len(d) for d in self.docs]
        self.avg_len = sum(self.doc_lens) / len(self.docs)
        self.df = Counter()
        for d in self.docs:
            for term in set(d):
                self.df[term] += 1
        self.n_docs = len(docs)

    @staticmethod
    def _tokenize(text):
        return re.findall(r"[a-z0-9]+", text.lower())

    def _idf(self, term):
        n_qualify = self.df.get(term, 0)
        return math.log((self.n_docs - n_qualify + 0.5) / (n_qualify + 0.5) + 1)

    def scores(self, query):
        q_terms = self._tokenize(query)
        out = np.zeros(self.n_docs)
        for i, doc in enumerate(self.docs):
            counts = Counter(doc)
            for term in q_terms:
                if term not in counts:
                    continue
                freq = counts[term]
                denom = freq + self.k1 * (1 - self.b + self.b * self.doc_lens[i] / self.avg_len)
                out[i] += self._idf(term) * (freq * (self.k1 + 1)) / denom
        return out

bm25 = BM25(corpus)
vectors = embed(corpus)

def rrf_fuse(query, k=60, top_k=3):
    bm25_ranks = np.argsort(bm25.scores(query))[::-1]
    q_vec = embed([query])[0]
    vec_sims = np.array([cosine_similarity(q_vec, v) for v in vectors])
    vec_ranks = np.argsort(vec_sims)[::-1]

    rrf_scores = np.zeros(len(corpus))
    for rank, idx in enumerate(bm25_ranks):
        rrf_scores[idx] += 1 / (k + rank)
    for rank, idx in enumerate(vec_ranks):
        rrf_scores[idx] += 1 / (k + rank)

    fused = np.argsort(rrf_scores)[::-1][:top_k]
    return [(corpus[i], rrf_scores[i]) for i in fused]

for chunk, score in rrf_fuse("what does error E-4521 mean"):
    print(f"{score:.4f}  {chunk}")


0.0333  Error code E-4521 means the payment gateway timed out; retry the charge after 30 seconds.
0.0328  Restarting the payment-gateway pod usually clears transient E-4521 errors.
0.0315  The French Revolution began in 1789 and led to the end of the monarchy.


**Reranking** goes one step further: hybrid search is cheap but approximate over the *whole* corpus,
so retrieve a wider candidate set (e.g. top 20) then rerank with a slower, more accurate cross-encoder
that scores the query and each candidate *jointly* (unlike embeddings, which score them independently).
In production this is usually a hosted reranker API (Cohere Rerank, Voyage, a local cross-encoder model)
— shown here as a stand-in that asks the chat model to score relevance directly, since it isn't worth
hand-rolling a cross-encoder for a demo.

In [3]:
def rerank(query, candidates, top_k=2):
    scored = []
    for chunk in candidates:
        prompt = f"""Rate how relevant this passage is to the query on a scale of 0-10.
Respond with only the number.

Query: {query}
Passage: {chunk}"""
        resp = client.models.generate_content(model=CHAT_MODEL, contents=prompt)
        try:
            score = float(resp.text.strip())
        except ValueError:
            score = 0.0
        scored.append((chunk, score))
    return sorted(scored, key=lambda x: x[1], reverse=True)[:top_k]

candidates = [c for c, _ in rrf_fuse("what does error E-4521 mean", top_k=4)]
for chunk, score in rerank("what does error E-4521 mean", candidates):
    print(f"{score:.1f}  {chunk}")


10.0  Error code E-4521 means the payment gateway timed out; retry the charge after 30 seconds.
10.0  Restarting the payment-gateway pod usually clears transient E-4521 errors.


### Chunking strategy tradeoffs

The fundamentals notebook used fixed-size character chunking. In production this is a real tuning
knob, not an afterthought:

| Strategy | Chunk size | Tradeoff |
|---|---|---|
| Fixed-size | Small (~100-200 tokens) | Precise retrieval, but context gets fragmented mid-sentence |
| Fixed-size | Large (~500-1000 tokens) | More context per chunk, but dilutes relevance — irrelevant text rides along |
| Recursive (paragraph/sentence-aware) | Variable | Respects natural boundaries, avoids mid-sentence cuts |
| Semantic (embedding-boundary) | Variable | Splits where meaning shifts, most accurate, most expensive to compute |

**Overlap** (e.g. 10-20% of chunk size) prevents a fact from being split exactly at a chunk boundary
and becoming unretrievable from either side. Too much overlap wastes storage and duplicates retrieval
results; too little risks losing facts that straddle a cut.

## 2. Evaluation & failure modes

You can't tune what you don't measure. Two layers matter:

- **Retrieval metrics** — did the right chunk get retrieved at all?
  - `Recall@k`: fraction of queries where the known-relevant chunk appears in the top k results.
  - `MRR` (Mean Reciprocal Rank): average of `1 / rank_of_first_relevant_result` — rewards relevant
    results appearing *near the top*, not just anywhere in top k.
- **Generation faithfulness** — did the model's answer actually come from the retrieved context, or
  did it hallucinate on top of it? Checked by asking a second LLM call whether every claim in the
  answer is supported by the provided context.

In [4]:
def recall_at_k(retriever_fn, eval_set, k=3):
    hits = 0
    for query, relevant_chunk in eval_set:
        retrieved = [c for c, _ in retriever_fn(query, top_k=k)]
        if relevant_chunk in retrieved:
            hits += 1
    return hits / len(eval_set)

def mrr(retriever_fn, eval_set, k=5):
    reciprocal_ranks = []
    for query, relevant_chunk in eval_set:
        retrieved = [c for c, _ in retriever_fn(query, top_k=k)]
        if relevant_chunk in retrieved:
            reciprocal_ranks.append(1 / (retrieved.index(relevant_chunk) + 1))
        else:
            reciprocal_ranks.append(0.0)
    return sum(reciprocal_ranks) / len(reciprocal_ranks)

# a hand-labeled eval set: (query, the chunk that should be retrieved)
eval_set = [
    ("what does error E-4521 mean", corpus[0]),
    ("how do I fix payment gateway timeouts", corpus[4]),
    ("can I get my money back", corpus[2]),
]

print("Recall@3:", recall_at_k(rrf_fuse, eval_set, k=3))
print("MRR:", mrr(rrf_fuse, eval_set, k=3))


Recall@3: 1.0


MRR: 0.6666666666666666


In [5]:
def precision_at_k(retriever_fn, eval_set, k=3):
    precisions = []
    for query, relevant_chunk in eval_set:
        retrieved = [c for c, _ in retriever_fn(query, top_k=k)]
        hits = sum(1 for c in retrieved if c == relevant_chunk)
        precisions.append(hits / k)
    return sum(precisions) / len(eval_set)

def ndcg_at_k(retriever_fn, eval_set, k=5):
    import math
    ndcgs = []
    for query, relevant_chunk in eval_set:
        retrieved = [c for c, _ in retriever_fn(query, top_k=k)]
        relevance = [1 if c == relevant_chunk else 0 for c in retrieved]
        dcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(relevance))
        ideal = sorted(relevance, reverse=True)
        idcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(ideal))
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)
    return sum(ndcgs) / len(eval_set)

print("Precision@3:", precision_at_k(rrf_fuse, eval_set, k=3))
print("NDCG@3:", ndcg_at_k(rrf_fuse, eval_set, k=3))

Precision@3: 0.3333333333333333


NDCG@3: 0.7539531690476383


In [6]:
def check_faithfulness(answer, context):
    prompt = f"""Context:
{context}

Answer: {answer}

Is every factual claim in the answer directly supported by the context above? Respond with only
YES or NO, then a one-sentence reason."""
    resp = client.models.generate_content(model=CHAT_MODEL, contents=prompt)
    return resp.text

context = corpus[0]
grounded_answer = "Error E-4521 means the payment gateway timed out."
hallucinated_answer = "Error E-4521 means the payment gateway timed out, and it was first introduced in v2.3."

print("Grounded:", check_faithfulness(grounded_answer, context))
print()
print("Hallucinated:", check_faithfulness(hallucinated_answer, context))


Grounded: YES. The answer directly states that E-4521 means the payment gateway timed out, which is explicitly provided in the context.



Hallucinated: NO. The context does not mention when error code E-4521 was first introduced.


### Named failure modes

- **Lost-in-the-middle** — LLMs attend more strongly to the start and end of a long context window;
  a relevant chunk buried in the middle of 10 stuffed chunks can get ignored even though it was
  retrieved correctly. Mitigate by keeping `top_k` small and reranking the most relevant chunk to the
  front (or the end) of the context.
- **Stale index** — a source document changes, but the vector store still holds the old embedding.
  The retriever confidently returns outdated information with no signal that anything is wrong.
- **Hallucinated citations** — the model cites a source that either wasn't retrieved or doesn't say
  what the citation claims. Faithfulness checking (above) catches this at the answer level.
- **Retrieval-generation mismatch** — retrieval found the right chunk, but the prompt template buries
  it, truncates it, or the model just ignores it in favor of its parametric knowledge.

## 3. Scaling & infra

`VectorStore` from the fundamentals notebook does a brute-force `O(n)` cosine scan over every vector
on every query — fine for a demo, unworkable past roughly 10k-100k vectors depending on latency budget.

**Approximate Nearest Neighbor (ANN) indexing** trades a small amount of recall for large speedups.
The dominant algorithm is **HNSW** (Hierarchical Navigable Small World): vectors are organized into a
multi-layer graph where top layers have long-range links for fast coarse navigation and bottom layers
have short-range links for fine-grained search — query time goes from `O(n)` to roughly `O(log n)`.

| | Exact (brute-force) | ANN (HNSW) |
|---|---|---|
| Recall | 100% | ~95-99%, tunable |
| Query latency at scale | Linear in corpus size | Near-constant |
| Build cost | None | Index build time + memory overhead |
| When to use | <10k vectors, or recall-critical | Anything larger |

**Where the vectors live in production** — pick based on what you already run and how much ops
overhead you want:
- **pgvector** — if you already run Postgres, add the extension, keep vectors next to your relational
  data, one less system to operate.
- **Chroma / Qdrant** — purpose-built vector DBs, easy to self-host, good default for a mid-size app.
- **Pinecone / Weaviate Cloud** — managed, scales without you operating the index, costs more per GB.

**Cost/latency knobs that actually move the needle**, roughly in order of impact:
1. `top_k` — every extra retrieved chunk adds prompt tokens (cost) and dilutes context (quality).
2. Rerank depth — reranking 50 candidates instead of 10 multiplies reranker cost ~5x for often
   marginal recall gains past the top ~20.
3. Embedding dimensionality — a 3072-dim embedding costs more to store and search than a 768-dim one;
   many providers offer truncated/quantized variants (e.g. Matryoshka embeddings) for a small quality
   hit.

## 4. Freshness & updates

An index is a cache of your documents at embedding time. Documents change; the index has to keep up.

- **Full rebuild** — simplest, re-embed and re-index everything on a schedule (nightly, hourly).
  Correct by construction, but wasteful and slow once the corpus is large, and stale between runs.
- **Incremental indexing** — embed and upsert only what changed. Requires tracking a stable chunk id
  (e.g. hash of `source_doc_id + chunk_offset`) so an update can find and replace the old vector
  instead of duplicating it.
- **Deletes** — don't just remove from the source; explicitly delete or tombstone the corresponding
  vectors, or a query can keep surfacing content that no longer exists anywhere else.
- **Point-in-time consistency** — if a document updates mid-query-batch, some queries may hit the old
  version and some the new one. Rarely worth solving with distributed transactions; usually solved
  by accepting eventual consistency and keeping the update-to-visible lag small.

In [7]:
class UpsertableStore:
    def __init__(self):
        self.ids = []
        self.chunks = {}
        self.vectors = {}

    def upsert(self, chunk_id, text):
        self.chunks[chunk_id] = text
        self.vectors[chunk_id] = embed([text])[0]
        if chunk_id not in self.ids:
            self.ids.append(chunk_id)

    def delete(self, chunk_id):
        self.ids.remove(chunk_id)
        del self.chunks[chunk_id]
        del self.vectors[chunk_id]

    def query(self, question, top_k=2):
        q_vec = embed([question])[0]
        sims = [(cid, cosine_similarity(q_vec, self.vectors[cid])) for cid in self.ids]
        sims.sort(key=lambda x: x[1], reverse=True)
        return [(self.chunks[cid], score) for cid, score in sims[:top_k]]

store = UpsertableStore()
store.upsert("policy-v1", "Refunds are accepted within 30 days of purchase.")
print("before update:", store.query("refund window")[0])

# the policy changes -- upsert with the same id replaces the stale vector, no duplicate
store.upsert("policy-v1", "Refunds are accepted within 60 days of purchase, extended for the holidays.")
print("after update:", store.query("refund window")[0])

store.delete("policy-v1")
print("remaining chunks:", store.ids)


before update: ('Refunds are accepted within 30 days of purchase.', np.float64(0.7464433262942389))


after update: ('Refunds are accepted within 60 days of purchase, extended for the holidays.', np.float64(0.7710605941577916))
remaining chunks: []
